# Per-Image Metrics (clean model)
Produces `comparison_metrics_60images_clean.csv` in the same schema as the original evidence file, using the clean retrained weights, plus split-level background-frame counts. Runtime: about 5-8 minutes on GPU. Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = "/content/drive/MyDrive/Uskudar Uni Master/YOLO Model klasor"
DS_ROOT = os.path.join(BASE, "Custom Dataset", "bogaz_v_1.v3i.yolov8")
TEST_IMAGES_PATH = os.path.join(BASE, "Test Dataset", "images")
TEST_LABELS_PATH = os.path.join(BASE, "Test Dataset", "labels")
CLEAN_MODEL_PATH = os.path.join(BASE, "runs", "bosphorus_yolo11s_clean", "weights", "best.pt")
OUTPUT_DIR = os.path.join(BASE, "Comparisons", "revision_outputs_clean")
assert os.path.exists(CLEAN_MODEL_PATH), "clean weights not found - run Clean_Retrain_Evaluation first"
print("weights found")

In [ ]:
!pip install -q -U ultralytics
from ultralytics import YOLO
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import numpy as np, time, glob
from PIL import Image

def parse_yolo_label(p, iw, ih):
    boxes = []
    if not os.path.exists(p):
        return boxes
    for line in open(p):
        parts = line.strip().split()
        if len(parts) == 5:
            xc, yc = float(parts[1])*iw, float(parts[2])*ih
            w, h = float(parts[3])*iw, float(parts[4])*ih
            boxes.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
        elif len(parts) > 5 and len(parts) % 2 == 1:
            xs = [float(v)*iw for v in parts[1::2]]; ys = [float(v)*ih for v in parts[2::2]]
            boxes.append([min(xs), min(ys), max(xs), max(ys)])
    return boxes

def iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    u = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter/u if u > 0 else 0

def match(preds, gts, thr=0.5):
    if not preds: return 0, 0, len(gts), []
    if not gts: return 0, len(preds), 0, []
    gm = [False]*len(gts); mi = []; tp = 0
    for p in preds:
        bi, bx = 0, -1
        for gi, g in enumerate(gts):
            if gm[gi]: continue
            v = iou(p, g)
            if v > bi: bi, bx = v, gi
        if bi >= thr and bx >= 0:
            gm[bx] = True; mi.append(bi); tp += 1
    return tp, len(preds)-tp, len(gts)-tp, mi

# background-frame counts per split
for split, lbl_dir in [('train', os.path.join(DS_ROOT,'train','labels')), ('valid', os.path.join(DS_ROOT,'valid','labels')), ('test', TEST_LABELS_PATH)]:
    empty = sum(1 for f in glob.glob(os.path.join(lbl_dir,'*.txt')) if not open(f).read().strip())
    print(f"{split}: {empty} background (vessel-free) label files")

In [ ]:
SPECS = {
    'YOLOv8n': ('yolov8n.pt', 640), 'YOLOv8s': ('yolov8s.pt', 640),
    'YOLOv11n': ('yolo11n.pt', 640), 'YOLOv11s': ('yolo11s.pt', 640),
    'Fine-tuned': (CLEAN_MODEL_PATH, 1088),
}
models = {k: YOLO(w) for k, (w, s) in SPECS.items()}
test_images = sorted(f for f in os.listdir(TEST_IMAGES_PATH) if f.lower().endswith(('.jpg','.jpeg','.png')))
rows = []
for img in test_images:
    pth = os.path.join(TEST_IMAGES_PATH, img)
    with Image.open(pth) as im: w, h = im.size
    gts = parse_yolo_label(os.path.join(TEST_LABELS_PATH, os.path.splitext(img)[0]+'.txt'), w, h)
    row = {'image': img, 'gt_count': len(gts)}
    for name, (wt, sz) in SPECS.items():
        t0 = time.time()
        rs = models[name].predict(source=pth, conf=0.25, imgsz=sz, classes=None, verbose=False)
        dt = (time.time()-t0)*1000
        boxes = [b.xyxy[0].cpu().numpy().tolist() for r in rs for b in r.boxes]
        confs = [float(b.conf[0]) for r in rs for b in r.boxes]
        tp, fp, fn, mi = match(boxes, gts)
        row.update({f'{name}_detections': len(boxes), f'{name}_tp': tp, f'{name}_fp': fp, f'{name}_fn': fn,
                    f'{name}_precision': tp/len(boxes) if boxes else 0.0, f'{name}_recall': tp/len(gts) if gts else 0.0,
                    f'{name}_matched_iou': float(np.mean(mi)) if mi else 0.0,
                    f'{name}_conf': float(np.mean(confs)) if confs else 0.0, f'{name}_time_ms': dt})
    rows.append(row)
import pandas as pd
df = pd.DataFrame(rows)
out = os.path.join(OUTPUT_DIR, 'comparison_metrics_60images_clean.csv')
df.to_csv(out, index=False)
print("saved", out)
ft_tp = df['Fine-tuned_tp'].sum(); gt = df['gt_count'].sum()
print(f"sanity: Fine-tuned TP {ft_tp}/{gt} (expect 287/321)")
zero_vanilla = ((df['YOLOv8n_tp']+df['YOLOv8s_tp']+df['YOLOv11n_tp']+df['YOLOv11s_tp'])==0).sum()
print(f"images where ALL vanilla models found nothing: {zero_vanilla}/60")
full_ft = (df['Fine-tuned_recall']==1.0).sum()
print(f"images with 100% fine-tuned recall: {full_ft}/60")